In [1]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append("..")

In [2]:
import pandas as pd
import numpy as np 
import src.config as cfg 

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("Project Root: ", cfg.ROOT_DIR)
print("Raw file exists: ", cfg.RAW_DATA_FILE.exists())

Project Root:  /home/kakarotbot/telco-churn-prediction
Raw file exists:  True


In [3]:
df = pd.read_csv(cfg.RAW_DATA_FILE)

print("Shape: ", df.shape)
print("\n--- dtypes ---")
print(df.dtypes)
print("\n--- head ---")
df.head()

Shape:  (7043, 21)

--- dtypes ---
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

--- head ---


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [6]:
# Schema Sanity Check

expected = set(
    [cfg.ID_COL, cfg.TARGET_COL]
    + cfg.NUMERIC_FEATURES
    + cfg.BINARY_FEATURES
    + cfg.PRE_ENCODED_BINARY_FEATURES
    + cfg.MULTICAT_FEATURES 
)

actual = set(df.columns)

print("Columns in CSV but not in config: ", actual - expected)
print("Columns in config but not in CSV: ", expected - actual)

# Duplicates across groups
from collections import Counter
all_lists = set(
    [cfg.ID_COL, cfg.TARGET_COL]
    + cfg.NUMERIC_FEATURES
    + cfg.BINARY_FEATURES
    + cfg.PRE_ENCODED_BINARY_FEATURES
    + cfg.MULTICAT_FEATURES 
)

dupes = [c for c, n in Counter(all_lists).items() if n > 1]
print("Duplicated across groups: ", dupes)

Columns in CSV but not in config:  set()
Columns in config but not in CSV:  set()
Duplicated across groups:  []


In [16]:
# Data-Quality Snapshot

# Missing values
print("--- missing values (count) ---")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Which columns are 'object' but might be numeric?
print("\n--- object columns ---")
print(df.select_dtypes(include="object").columns.tolist())

# Target balance
print("\n--- target distribution ---")
print(df[cfg.TARGET_COL].value_counts(normalize=True) * 100)

--- missing values (count) ---
Series([], dtype: int64)

--- object columns ---
['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges', 'Churn']

--- target distribution ---
Churn
No     73.463013
Yes    26.536987
Name: proportion, dtype: float64


/tmp/ipykernel_14662/182265108.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(df.select_dtypes(include="object").columns.tolist())


In [40]:
for col in df.select_dtypes(include="object").columns:
    converted = pd.to_numeric(df[col], errors="coerce")
    print(col, "->", converted.notna().sum(), "numeric values")

customerID -> 0 numeric values
gender -> 0 numeric values
Partner -> 0 numeric values
Dependents -> 0 numeric values
PhoneService -> 0 numeric values
MultipleLines -> 0 numeric values
InternetService -> 0 numeric values
OnlineSecurity -> 0 numeric values
OnlineBackup -> 0 numeric values
DeviceProtection -> 0 numeric values
TechSupport -> 0 numeric values
StreamingTV -> 0 numeric values
StreamingMovies -> 0 numeric values
Contract -> 0 numeric values
PaperlessBilling -> 0 numeric values
PaymentMethod -> 0 numeric values
TotalCharges -> 7032 numeric values
Churn -> 0 numeric values


/tmp/ipykernel_14662/1134579033.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


In [22]:
blank = (df["TotalCharges"].astype(str).str.strip() == "").sum()
coerced_nan = pd.to_numeric(df["TotalCharges"], errors="coerce").isna().sum()
print(f"Blank strings: {blank} | NaN after coerce: {coerced_nan}")

# Who are these rows?
mask = pd.to_numeric(df["TotalCharges"], errors="coerce").isna()
print(df.loc[mask, ["tenure", "MonthlyCharges", "TotalCharges", "Churn"]])

Blank strings: 11 | NaN after coerce: 11
      tenure  MonthlyCharges TotalCharges Churn
488        0           52.55                 No
753        0           20.25                 No
936        0           80.85                 No
1082       0           25.75                 No
1340       0           56.05                 No
3331       0           19.85                 No
3826       0           25.35                 No
4380       0           20.00                 No
5218       0           19.70                 No
6670       0           73.35                 No
6754       0           61.90                 No


In [41]:
# finding hidden missingness in object cols
for col in df.select_dtypes(include="object").columns:
    n_blank = (df[col].astype(str).str.strip() == "").sum()
    n_numeric_coerced = pd.to_numeric(n_blank, errors="coerce")
    if n_blank or n_numeric_coerced > 0:
        print(f"{col}: blanks={n_blank}, numeric-parseable={n_numeric_coerced} ")


TotalCharges: blanks=11, numeric-parseable=11 


/tmp/ipykernel_14662/3316664123.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


### Data Quality Notes
- Shape: 7043 × 21
- Target balance: No 73.46% / Yes 26.54% → mild class imbalance (~1:2.8)
- Missing: TotalCharges — 11 blank strings for tenure=0 new customers
- Object cols: 18 (customerID, 16 categorical, TotalCharges, Churn)
- Config schema check: PASS (after adding SeniorCitizen as pre-encoded binary)